In [1]:
from langchain_openai import ChatOpenAI

In [2]:
llm = ChatOpenAI()

llm.invoke("HOli")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-Den2E1aSioyti7a5jGzLLoRIpzm2o', 'finish_reason': 'stop', 'logprobs': None}, id='run--a45cba37-0b22-4cde-8050-04bb9aa1129f-0', usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [3]:
!pip show tsf-rag

Name: tsf-rag
Version: 0.1.0
Summary: AsyncMultiQuery TSF-RAG System for TSF Physical Stability Evaluation
Home-page: https://github.com/GbrlOl/async-multi-query-tsf-rag
Author: 
Author-email: Gabriel Olmos <gabriel.olmos@pucv.cl>
License: GPL-3.0-or-later
Location: c:\users\gol_m\anaconda3\envs\paper_geotecnia\lib\site-packages
Editable project location: C:\Users\gol_m\OneDrive\Desktop\eswa
Requires: 
Required-by: 


# **RAG System Evaluation**

In [1]:
from evaluation_rag import auto_evaluation_rag_v2
from llms_modules import LLMEvaluator

## **1. Naive RAG System Evaluation**

In [2]:
from rag import NaiveRAG

c:\Users\gol_m\anaconda3\envs\paper_geotecnia\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


resource module not available on Windows


### **1.1 Sand Tailing Dams (STD)**

In [3]:
import os
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

N_ITERATIONS = 20

OUTPUT_FOLDER = "NaiveRAG_TSF_STD_Evaluation"

PATH_GROUND_TRUTH   = "../src/evaluation_rag/rag_ground_truth/01_STD_Tranque_GT.xlsx"
PATH_VECTOR_STORAGE = "../src/evaluation_retrieval/vector_storage/01_cerro_negro_tranque/01_cerro_negro_nomic"
EMBEDDING_TYPE      = "nomic"
EMBEDDING_MODEL     = "nomic-embed-text-v1.5"

CONFIDENCE   = 0.95
OUTPUT_STATS = "NaiveRAG_TSF_STD_Statistics.xlsx"

def get_last_iteration(folder: Path) -> int:
    if not folder.exists():
        return 0
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    numbers = []
    for f in folder.iterdir():
        match = pattern.match(f.name)
        if match:
            numbers.append(int(match.group(1)))
    return max(numbers) if numbers else 0


def parse_accuracy(value) -> float:
    if isinstance(value, (int, float)):
        v = float(value)
        return v / 100 if v > 1 else v
    cleaned = str(value).replace("%", "").strip()
    try:
        v = float(cleaned)
        return v / 100 if v > 1 else v
    except ValueError:
        return float("nan")


def load_iteration(path: Path) -> dict:
    xl = pd.read_excel(path, sheet_name="resultado final")
    xl.columns = xl.columns.str.strip()
    result = {}
    for _, row in xl.iterrows():
        matriz = str(row["Matriz"]).strip()
        acc    = parse_accuracy(row["Accuracy"])
        result[matriz] = acc
    return result


def get_iteration_files(folder: Path) -> list:
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    files = []
    for f in folder.iterdir():
        if pattern.match(f.name):
            files.append(f)
    return sorted(files, key=lambda f: int(pattern.match(f.name).group(1)))


def wilson_ci(p: float, n: int, confidence: float = 0.95) -> tuple:
    if n == 0 or np.isnan(p):
        return (float("nan"), float("nan"))
    z           = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre      = (p + z**2 / (2 * n)) / denominator
    margin      = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    lower = max(0.0, centre - margin)
    upper = min(1.0, centre + margin)
    return (lower, upper)

pipeline_rag = NaiveRAG(
    path_vector_storage=PATH_VECTOR_STORAGE,
    embedding_type=EMBEDDING_TYPE,
    embedding_model_name=EMBEDDING_MODEL,
)
llm_evaluador = LLMEvaluator()

folder = Path(OUTPUT_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

last  = get_last_iteration(folder)
start = last + 1
end   = last + N_ITERATIONS

print(f"Iteraciones previas encontradas : {last}")
print(f"Corriendo iteraciones           : {start} → {end}")
print(f"Carpeta de salida               : {folder.resolve()}\n")

for i in range(start, end + 1):
    name_excel = folder / f"iteration_{i:03d}.xlsx"
    print(f"{'='*55}")
    print(f"  Iteración {i:03d} de {end:03d} → {name_excel.name}")
    print(f"{'='*55}")

    inicio = time.time()

    auto_evaluation_rag_v2(
        path_ground_truth=PATH_GROUND_TRUTH,
        name_excel=str(name_excel),
        pipeline_rag=pipeline_rag,
        llm_evaluador=llm_evaluador,
    )

    elapsed = time.time() - inicio
    print(f"  ✓ Completada en {elapsed:.1f}s ({elapsed/60:.2f} min)\n")

print("Todas las iteraciones completadas.")

Iteraciones previas encontradas : 10
Corriendo iteraciones           : 11 → 30
Carpeta de salida               : C:\Users\gol_m\OneDrive\Desktop\eswa\experiments\NaiveRAG_TSF_STD_Evaluation

  Iteración 011 de 030 → iteration_011.xlsx


Evaluación Matriz:   0%|          | 0/8 [00:00<?, ?it/s]

Matriz 1


Evaluación Matriz:  12%|█▎        | 1/8 [00:24<02:48, 24.04s/it]

Matriz 2


...

Matriz 7


Evaluación Matriz:  88%|████████▊ | 7/8 [01:02<00:08,  8.48s/it]

Matriz 8


Evaluación Matriz: 100%|██████████| 8/8 [01:19<00:00,  9.89s/it]

  ✓ Completada en 79.3s (1.32 min)

Todas las iteraciones completadas.


In [6]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

# Cargar todas las iteraciones
iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

# Construir DataFrame de detalle
all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

# Calcular estadísticas por matriz
stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

# Formatear detalle como porcentajes legibles
df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

# Mostrar en notebook
print("\nEstadísticas por matriz:")
display(df_stats)

# Exportar Excel
with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 10
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,10,50.0,0.00,23.66,76.34
1,Matriz 2,10,60.0,14.06,31.27,83.18
2,Matriz 4,10,100.0,0.00,72.25,100.00
3,Matriz 6,10,100.0,0.00,72.25,100.00
4,Matriz 7,10,35.0,24.15,13.69,64.63
5,Matriz 8,10,100.0,0.00,72.25,100.00
6,Total (matrices evaluadas),10,65.0,7.07,35.37,86.31



✓ Estadísticas exportadas → NaiveRAG_TSF_STD_Statistics.xlsx


In [4]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

# Cargar todas las iteraciones
iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

# Construir DataFrame de detalle
all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

# Calcular estadísticas por matriz
stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

# Formatear detalle como porcentajes legibles
df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

# Mostrar en notebook
print("\nEstadísticas por matriz:")
display(df_stats)

# Exportar Excel
with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 30
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx
  iteration_021.xlsx
  iteration_022.xlsx
  iteration_023.xlsx
  iteration_024.xlsx
  iteration_025.xlsx
  iteration_026.xlsx
  iteration_027.xlsx
  iteration_028.xlsx
  iteration_029.xlsx
  iteration_030.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,30,50.0,0.00,33.15,66.85
1,Matriz 2,30,60.0,13.56,42.32,75.41
2,Matriz 4,30,100.0,0.00,88.65,100.00
3,Matriz 6,30,100.0,0.00,88.65,100.00
4,Matriz 7,30,20.0,24.91,9.51,37.31
5,Matriz 8,30,100.0,0.00,88.65,100.00
6,Total (matrices evaluadas),30,62.0,6.64,44.23,77.05



✓ Estadísticas exportadas → NaiveRAG_TSF_STD_Statistics.xlsx


### **1.2 Filtered Tailings Deposit (FTD)**

In [4]:
import os
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

N_ITERATIONS = 30

OUTPUT_FOLDER = "NaiveRAG_TSF_FTD_Evaluation"

PATH_GROUND_TRUTH = "../src/evaluation_rag/rag_ground_truth/02_FTD_Filtrado_GT.xlsx"
PATH_VECTOR_STORAGE = "../src/evaluation_retrieval/vector_storage/02_kozan_filtrado/02_kozan_nomic"
EMBEDDING_TYPE      = "nomic"
EMBEDDING_MODEL     = "nomic-embed-text-v1.5"

CONFIDENCE   = 0.95
OUTPUT_STATS = "NaiveRAG_TSF_FTD_Statistics.xlsx"

def get_last_iteration(folder: Path) -> int:
    if not folder.exists():
        return 0
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    numbers = []
    for f in folder.iterdir():
        match = pattern.match(f.name)
        if match:
            numbers.append(int(match.group(1)))
    return max(numbers) if numbers else 0


def parse_accuracy(value) -> float:
    if isinstance(value, (int, float)):
        v = float(value)
        return v / 100 if v > 1 else v
    cleaned = str(value).replace("%", "").strip()
    try:
        v = float(cleaned)
        return v / 100 if v > 1 else v
    except ValueError:
        return float("nan")


def load_iteration(path: Path) -> dict:
    xl = pd.read_excel(path, sheet_name="resultado final")
    xl.columns = xl.columns.str.strip()
    result = {}
    for _, row in xl.iterrows():
        matriz = str(row["Matriz"]).strip()
        acc    = parse_accuracy(row["Accuracy"])
        result[matriz] = acc
    return result


def get_iteration_files(folder: Path) -> list:
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    files = []
    for f in folder.iterdir():
        if pattern.match(f.name):
            files.append(f)
    return sorted(files, key=lambda f: int(pattern.match(f.name).group(1)))


def wilson_ci(p: float, n: int, confidence: float = 0.95) -> tuple:
    if n == 0 or np.isnan(p):
        return (float("nan"), float("nan"))
    z           = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre      = (p + z**2 / (2 * n)) / denominator
    margin      = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    lower = max(0.0, centre - margin)
    upper = min(1.0, centre + margin)
    return (lower, upper)

pipeline_rag = NaiveRAG(
    path_vector_storage=PATH_VECTOR_STORAGE,
    embedding_type=EMBEDDING_TYPE,
    embedding_model_name=EMBEDDING_MODEL,
)
llm_evaluador = LLMEvaluator()

folder = Path(OUTPUT_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

last  = get_last_iteration(folder)
start = last + 1
end   = last + N_ITERATIONS

print(f"Iteraciones previas encontradas : {last}")
print(f"Corriendo iteraciones           : {start} → {end}")
print(f"Carpeta de salida               : {folder.resolve()}\n")

for i in range(start, end + 1):
    name_excel = folder / f"iteration_{i:03d}.xlsx"
    print(f"{'='*55}")
    print(f"  Iteración {i:03d} de {end:03d} → {name_excel.name}")
    print(f"{'='*55}")

    inicio = time.time()

    auto_evaluation_rag_v2(
        path_ground_truth=PATH_GROUND_TRUTH,
        name_excel=str(name_excel),
        pipeline_rag=pipeline_rag,
        llm_evaluador=llm_evaluador,
    )

    elapsed = time.time() - inicio
    print(f"  ✓ Completada en {elapsed:.1f}s ({elapsed/60:.2f} min)\n")

print("Todas las iteraciones completadas.")

Iteraciones previas encontradas : 0
Corriendo iteraciones           : 1 → 30
Carpeta de salida               : C:\Users\gol_m\OneDrive\Desktop\eswa\experiments\NaiveRAG_TSF_FTD_Evaluation

  Iteración 001 de 030 → iteration_001.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [02:19<00:00, 17.41s/it]


  ✓ Completada en 139.6s (2.33 min)

  Iteración 002 de 030 → iteration_002.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:50<00:00, 13.86s/it]


  ✓ Completada en 111.1s (1.85 min)

  Iteración 003 de 030 → iteration_003.xlsx


...

  ✓ Completada en 96.2s (1.60 min)

  Iteración 029 de 030 → iteration_029.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:38<00:00, 12.33s/it]


  ✓ Completada en 98.8s (1.65 min)

  Iteración 030 de 030 → iteration_030.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:42<00:00, 12.77s/it]

  ✓ Completada en 102.3s (1.71 min)

Todas las iteraciones completadas.


In [5]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

# Cargar todas las iteraciones
iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

# Construir DataFrame de detalle
all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

# Calcular estadísticas por matriz
stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

# Formatear detalle como porcentajes legibles
df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

# Mostrar en notebook
print("\nEstadísticas por matriz:")
display(df_stats)

# Exportar Excel
with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 30
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx
  iteration_021.xlsx
  iteration_022.xlsx
  iteration_023.xlsx
  iteration_024.xlsx
  iteration_025.xlsx
  iteration_026.xlsx
  iteration_027.xlsx
  iteration_028.xlsx
  iteration_029.xlsx
  iteration_030.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,30,100.00,0.00,88.65,100.00
1,Matriz 2,30,29.00,4.03,15.91,46.86
2,Matriz 4,30,100.00,0.00,88.65,100.00
3,Matriz 6,30,100.00,0.00,88.65,100.00
4,Matriz 7,30,36.67,22.49,21.87,54.49
5,Matriz 8,30,100.00,0.00,88.65,100.00
6,Total (matrices evaluadas),30,47.71,3.84,31.14,64.80



✓ Estadísticas exportadas → NaiveRAG_TSF_FTD_Statistics.xlsx


### **1.3 Thickened Tailings Deposit (TTD)**

In [6]:
import os
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

N_ITERATIONS = 30

OUTPUT_FOLDER = "NaiveRAG_TSF_TTD_Evaluation"

PATH_GROUND_TRUTH = "../src/evaluation_rag/rag_ground_truth/03_TTD_Espesado_GT.xlsx"
PATH_VECTOR_STORAGE = "../src/evaluation_retrieval/vector_storage/03_cenizas_espesado/03_cenizas_nomic"
EMBEDDING_TYPE      = "nomic"
EMBEDDING_MODEL     = "nomic-embed-text-v1.5"

CONFIDENCE   = 0.95
OUTPUT_STATS = "NaiveRAG_TSF_TTD_Statistics.xlsx"

def get_last_iteration(folder: Path) -> int:
    if not folder.exists():
        return 0
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    numbers = []
    for f in folder.iterdir():
        match = pattern.match(f.name)
        if match:
            numbers.append(int(match.group(1)))
    return max(numbers) if numbers else 0


def parse_accuracy(value) -> float:
    if isinstance(value, (int, float)):
        v = float(value)
        return v / 100 if v > 1 else v
    cleaned = str(value).replace("%", "").strip()
    try:
        v = float(cleaned)
        return v / 100 if v > 1 else v
    except ValueError:
        return float("nan")


def load_iteration(path: Path) -> dict:
    xl = pd.read_excel(path, sheet_name="resultado final")
    xl.columns = xl.columns.str.strip()
    result = {}
    for _, row in xl.iterrows():
        matriz = str(row["Matriz"]).strip()
        acc    = parse_accuracy(row["Accuracy"])
        result[matriz] = acc
    return result


def get_iteration_files(folder: Path) -> list:
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    files = []
    for f in folder.iterdir():
        if pattern.match(f.name):
            files.append(f)
    return sorted(files, key=lambda f: int(pattern.match(f.name).group(1)))


def wilson_ci(p: float, n: int, confidence: float = 0.95) -> tuple:
    if n == 0 or np.isnan(p):
        return (float("nan"), float("nan"))
    z           = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre      = (p + z**2 / (2 * n)) / denominator
    margin      = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    lower = max(0.0, centre - margin)
    upper = min(1.0, centre + margin)
    return (lower, upper)

pipeline_rag = NaiveRAG(
    path_vector_storage=PATH_VECTOR_STORAGE,
    embedding_type=EMBEDDING_TYPE,
    embedding_model_name=EMBEDDING_MODEL,
)
llm_evaluador = LLMEvaluator()

folder = Path(OUTPUT_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

last  = get_last_iteration(folder)
start = last + 1
end   = last + N_ITERATIONS

print(f"Iteraciones previas encontradas : {last}")
print(f"Corriendo iteraciones           : {start} → {end}")
print(f"Carpeta de salida               : {folder.resolve()}\n")

for i in range(start, end + 1):
    name_excel = folder / f"iteration_{i:03d}.xlsx"
    print(f"{'='*55}")
    print(f"  Iteración {i:03d} de {end:03d} → {name_excel.name}")
    print(f"{'='*55}")

    inicio = time.time()

    auto_evaluation_rag_v2(
        path_ground_truth=PATH_GROUND_TRUTH,
        name_excel=str(name_excel),
        pipeline_rag=pipeline_rag,
        llm_evaluador=llm_evaluador,
    )

    elapsed = time.time() - inicio
    print(f"  ✓ Completada en {elapsed:.1f}s ({elapsed/60:.2f} min)\n")

print("Todas las iteraciones completadas.")

Iteraciones previas encontradas : 0
Corriendo iteraciones           : 1 → 30
Carpeta de salida               : C:\Users\gol_m\OneDrive\Desktop\eswa\experiments\NaiveRAG_TSF_TTD_Evaluation

  Iteración 001 de 030 → iteration_001.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:43<00:00, 12.94s/it]


  ✓ Completada en 103.7s (1.73 min)

  Iteración 002 de 030 → iteration_002.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:40<00:00, 12.62s/it]


  ✓ Completada en 101.1s (1.69 min)

  Iteración 003 de 030 → iteration_003.xlsx


...

  ✓ Completada en 67.2s (1.12 min)

  Iteración 029 de 030 → iteration_029.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:13<00:00,  9.13s/it]


  ✓ Completada en 73.2s (1.22 min)

  Iteración 030 de 030 → iteration_030.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:07<00:00,  8.38s/it]

  ✓ Completada en 67.2s (1.12 min)

Todas las iteraciones completadas.


In [7]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

# Cargar todas las iteraciones
iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

# Construir DataFrame de detalle
all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

# Calcular estadísticas por matriz
stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

# Formatear detalle como porcentajes legibles
df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

# Mostrar en notebook
print("\nEstadísticas por matriz:")
display(df_stats)

# Exportar Excel
with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 30
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx
  iteration_021.xlsx
  iteration_022.xlsx
  iteration_023.xlsx
  iteration_024.xlsx
  iteration_025.xlsx
  iteration_026.xlsx
  iteration_027.xlsx
  iteration_028.xlsx
  iteration_029.xlsx
  iteration_030.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,30,100.00,0.00,88.65,100.00
1,Matriz 2,30,33.33,0.00,19.23,51.22
2,Matriz 4,30,100.00,0.00,88.65,100.00
3,Matriz 6,30,100.00,0.00,88.65,100.00
4,Matriz 7,30,98.33,9.13,85.87,99.83
5,Matriz 8,30,10.00,30.51,3.46,25.62
6,Total (matrices evaluadas),30,58.89,3.04,41.27,74.49



✓ Estadísticas exportadas → NaiveRAG_TSF_TTD_Statistics.xlsx


### **1.4 Tailing Embankments (TE)**

In [3]:
import os
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

N_ITERATIONS = 30

OUTPUT_FOLDER = "NaiveRAG_TSF_TE_Evaluation"

PATH_GROUND_TRUTH = "../src/evaluation_rag/rag_ground_truth/04_TE_Embalse_GT.xlsx"
PATH_VECTOR_STORAGE = "../src/evaluation_retrieval/vector_storage/04_enami_embalse/04_enami_nomic"
EMBEDDING_TYPE      = "nomic"
EMBEDDING_MODEL     = "nomic-embed-text-v1.5"

CONFIDENCE   = 0.95
OUTPUT_STATS = "NaiveRAG_TSF_TE_Statistics.xlsx"

def get_last_iteration(folder: Path) -> int:
    if not folder.exists():
        return 0
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    numbers = []
    for f in folder.iterdir():
        match = pattern.match(f.name)
        if match:
            numbers.append(int(match.group(1)))
    return max(numbers) if numbers else 0


def parse_accuracy(value) -> float:
    if isinstance(value, (int, float)):
        v = float(value)
        return v / 100 if v > 1 else v
    cleaned = str(value).replace("%", "").strip()
    try:
        v = float(cleaned)
        return v / 100 if v > 1 else v
    except ValueError:
        return float("nan")


def load_iteration(path: Path) -> dict:
    xl = pd.read_excel(path, sheet_name="resultado final")
    xl.columns = xl.columns.str.strip()
    result = {}
    for _, row in xl.iterrows():
        matriz = str(row["Matriz"]).strip()
        acc    = parse_accuracy(row["Accuracy"])
        result[matriz] = acc
    return result


def get_iteration_files(folder: Path) -> list:
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    files = []
    for f in folder.iterdir():
        if pattern.match(f.name):
            files.append(f)
    return sorted(files, key=lambda f: int(pattern.match(f.name).group(1)))


def wilson_ci(p: float, n: int, confidence: float = 0.95) -> tuple:
    if n == 0 or np.isnan(p):
        return (float("nan"), float("nan"))
    z           = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre      = (p + z**2 / (2 * n)) / denominator
    margin      = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    lower = max(0.0, centre - margin)
    upper = min(1.0, centre + margin)
    return (lower, upper)

pipeline_rag = NaiveRAG(
    path_vector_storage=PATH_VECTOR_STORAGE,
    embedding_type=EMBEDDING_TYPE,
    embedding_model_name=EMBEDDING_MODEL,
)
llm_evaluador = LLMEvaluator()

folder = Path(OUTPUT_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

last  = get_last_iteration(folder)
start = last + 1
end   = last + N_ITERATIONS

print(f"Iteraciones previas encontradas : {last}")
print(f"Corriendo iteraciones           : {start} → {end}")
print(f"Carpeta de salida               : {folder.resolve()}\n")

for i in range(start, end + 1):
    name_excel = folder / f"iteration_{i:03d}.xlsx"
    print(f"{'='*55}")
    print(f"  Iteración {i:03d} de {end:03d} → {name_excel.name}")
    print(f"{'='*55}")

    inicio = time.time()

    auto_evaluation_rag_v2(
        path_ground_truth=PATH_GROUND_TRUTH,
        name_excel=str(name_excel),
        pipeline_rag=pipeline_rag,
        llm_evaluador=llm_evaluador,
    )

    elapsed = time.time() - inicio
    print(f"  ✓ Completada en {elapsed:.1f}s ({elapsed/60:.2f} min)\n")

print("Todas las iteraciones completadas.")

Iteraciones previas encontradas : 0
Corriendo iteraciones           : 1 → 30
Carpeta de salida               : C:\Users\gol_m\OneDrive\Desktop\eswa\experiments\NaiveRAG_TSF_TE_Evaluation

  Iteración 001 de 030 → iteration_001.xlsx


Evaluación Matriz:   0%|          | 0/8 [00:00<?, ?it/s]

Matriz 1


Evaluación Matriz:  12%|█▎        | 1/8 [00:11<01:23, 11.95s/it]

Matriz 2


...

Matriz 7


Evaluación Matriz:  88%|████████▊ | 7/8 [00:48<00:07,  7.17s/it]

Matriz 8


Evaluación Matriz: 100%|██████████| 8/8 [00:56<00:00,  7.07s/it]

  ✓ Completada en 56.7s (0.95 min)

Todas las iteraciones completadas.


In [4]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

# Cargar todas las iteraciones
iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

# Construir DataFrame de detalle
all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

# Calcular estadísticas por matriz
stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

# Formatear detalle como porcentajes legibles
df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

# Mostrar en notebook
print("\nEstadísticas por matriz:")
display(df_stats)

# Exportar Excel
with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 30
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx
  iteration_021.xlsx
  iteration_022.xlsx
  iteration_023.xlsx
  iteration_024.xlsx
  iteration_025.xlsx
  iteration_026.xlsx
  iteration_027.xlsx
  iteration_028.xlsx
  iteration_029.xlsx
  iteration_030.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,30,0.0,0.0,0.00,11.35
1,Matriz 2,30,75.0,0.0,57.30,87.02
2,Matriz 4,30,100.0,0.0,88.65,100.00
3,Matriz 6,30,0.0,0.0,0.00,11.35
4,Matriz 7,30,50.0,0.0,33.15,66.85
5,Matriz 8,30,100.0,0.0,88.65,100.00
6,Total (matrices evaluadas),30,60.0,0.0,42.32,75.41



✓ Estadísticas exportadas → NaiveRAG_TSF_TE_Statistics.xlsx


### **1.5 Paste Tailings Deposits (PTD)**

In [3]:
import os
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

N_ITERATIONS = 30

OUTPUT_FOLDER = "NaiveRAG_TSF_PTD_Evaluation"

PATH_GROUND_TRUTH = "../src/evaluation_rag/rag_ground_truth/05_PTD_EnPasta_GT.xlsx"
PATH_VECTOR_STORAGE = "../src/evaluation_retrieval/vector_storage/05_florida_pasta/05_florida_nomic"
EMBEDDING_TYPE      = "nomic"
EMBEDDING_MODEL     = "nomic-embed-text-v1.5"

CONFIDENCE   = 0.95
OUTPUT_STATS = "NaiveRAG_TSF_PTD_Statistics.xlsx"

def get_last_iteration(folder: Path) -> int:
    if not folder.exists():
        return 0
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    numbers = []
    for f in folder.iterdir():
        match = pattern.match(f.name)
        if match:
            numbers.append(int(match.group(1)))
    return max(numbers) if numbers else 0


def parse_accuracy(value) -> float:
    if isinstance(value, (int, float)):
        v = float(value)
        return v / 100 if v > 1 else v
    cleaned = str(value).replace("%", "").strip()
    try:
        v = float(cleaned)
        return v / 100 if v > 1 else v
    except ValueError:
        return float("nan")


def load_iteration(path: Path) -> dict:
    xl = pd.read_excel(path, sheet_name="resultado final")
    xl.columns = xl.columns.str.strip()
    result = {}
    for _, row in xl.iterrows():
        matriz = str(row["Matriz"]).strip()
        acc    = parse_accuracy(row["Accuracy"])
        result[matriz] = acc
    return result


def get_iteration_files(folder: Path) -> list:
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    files = []
    for f in folder.iterdir():
        if pattern.match(f.name):
            files.append(f)
    return sorted(files, key=lambda f: int(pattern.match(f.name).group(1)))


def wilson_ci(p: float, n: int, confidence: float = 0.95) -> tuple:
    if n == 0 or np.isnan(p):
        return (float("nan"), float("nan"))
    z           = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre      = (p + z**2 / (2 * n)) / denominator
    margin      = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    lower = max(0.0, centre - margin)
    upper = min(1.0, centre + margin)
    return (lower, upper)

pipeline_rag = NaiveRAG(
    path_vector_storage=PATH_VECTOR_STORAGE,
    embedding_type=EMBEDDING_TYPE,
    embedding_model_name=EMBEDDING_MODEL,
)
llm_evaluador = LLMEvaluator()

folder = Path(OUTPUT_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

last  = get_last_iteration(folder)
start = last + 1
end   = last + N_ITERATIONS

print(f"Iteraciones previas encontradas : {last}")
print(f"Corriendo iteraciones           : {start} → {end}")
print(f"Carpeta de salida               : {folder.resolve()}\n")

for i in range(start, end + 1):
    name_excel = folder / f"iteration_{i:03d}.xlsx"
    print(f"{'='*55}")
    print(f"  Iteración {i:03d} de {end:03d} → {name_excel.name}")
    print(f"{'='*55}")

    inicio = time.time()

    auto_evaluation_rag_v2(
        path_ground_truth=PATH_GROUND_TRUTH,
        name_excel=str(name_excel),
        pipeline_rag=pipeline_rag,
        llm_evaluador=llm_evaluador,
    )

    elapsed = time.time() - inicio
    print(f"  ✓ Completada en {elapsed:.1f}s ({elapsed/60:.2f} min)\n")

print("Todas las iteraciones completadas.")

Iteraciones previas encontradas : 0
Corriendo iteraciones           : 1 → 30
Carpeta de salida               : C:\Users\gol_m\OneDrive\Desktop\eswa\experiments\NaiveRAG_TSF_PTD_Evaluation

  Iteración 001 de 030 → iteration_001.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [02:20<00:00, 17.51s/it]


  ✓ Completada en 140.4s (2.34 min)

  Iteración 002 de 030 → iteration_002.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:45<00:00, 13.16s/it]


  ✓ Completada en 105.5s (1.76 min)

  Iteración 003 de 030 → iteration_003.xlsx


...

  ✓ Completada en 84.9s (1.42 min)

  Iteración 029 de 030 → iteration_029.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:10<00:00,  8.86s/it]


  ✓ Completada en 71.1s (1.18 min)

  Iteración 030 de 030 → iteration_030.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:10<00:00,  8.85s/it]

  ✓ Completada en 70.9s (1.18 min)

Todas las iteraciones completadas.


In [4]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

# Cargar todas las iteraciones
iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

# Construir DataFrame de detalle
all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

# Calcular estadísticas por matriz
stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

# Formatear detalle como porcentajes legibles
df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

# Mostrar en notebook
print("\nEstadísticas por matriz:")
display(df_stats)

# Exportar Excel
with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 30
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx
  iteration_021.xlsx
  iteration_022.xlsx
  iteration_023.xlsx
  iteration_024.xlsx
  iteration_025.xlsx
  iteration_026.xlsx
  iteration_027.xlsx
  iteration_028.xlsx
  iteration_029.xlsx
  iteration_030.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,30,100.00,0.00,88.65,100.00
1,Matriz 2,30,47.78,5.76,31.20,64.86
2,Matriz 4,30,100.00,0.00,88.65,100.00
3,Matriz 6,30,96.67,18.26,83.33,99.41
4,Matriz 7,30,50.00,0.00,33.15,66.85
5,Matriz 8,30,100.00,0.00,88.65,100.00
6,Total (matrices evaluadas),30,65.28,3.84,47.41,79.68



✓ Estadísticas exportadas → NaiveRAG_TSF_PTD_Statistics.xlsx


## **2. Advanced RAG System Evaluation**

In [2]:
from rag import AdvancedRAG

c:\Users\gol_m\anaconda3\envs\paper_geotecnia\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


resource module not available on Windows


### **2.1 Sand Tailing Dams (STD)**

In [3]:
import os
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

N_ITERATIONS = 30

OUTPUT_FOLDER = "02_AdvancedRAG_TSF_STD_Evaluation"

PATH_GROUND_TRUTH   = "../src/evaluation_rag/rag_ground_truth/01_STD_Tranque_GT.xlsx"
PATH_VECTOR_STORAGE = "../src/evaluation_retrieval/vector_storage/01_cerro_negro_tranque/01_cerro_negro_nomic"
EMBEDDING_TYPE      = "nomic"
EMBEDDING_MODEL     = "nomic-embed-text-v1.5"

CROSS_ENCODER_MODEL_NAME="cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_N = 5

CONFIDENCE   = 0.95
OUTPUT_STATS = "02_AdvancedRAG_TSF_STD_Statistics.xlsx"

def get_last_iteration(folder: Path) -> int:
    if not folder.exists():
        return 0
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    numbers = []
    for f in folder.iterdir():
        match = pattern.match(f.name)
        if match:
            numbers.append(int(match.group(1)))
    return max(numbers) if numbers else 0


def parse_accuracy(value) -> float:
    if isinstance(value, (int, float)):
        v = float(value)
        return v / 100 if v > 1 else v
    cleaned = str(value).replace("%", "").strip()
    try:
        v = float(cleaned)
        return v / 100 if v > 1 else v
    except ValueError:
        return float("nan")


def load_iteration(path: Path) -> dict:
    xl = pd.read_excel(path, sheet_name="resultado final")
    xl.columns = xl.columns.str.strip()
    result = {}
    for _, row in xl.iterrows():
        matriz = str(row["Matriz"]).strip()
        acc    = parse_accuracy(row["Accuracy"])
        result[matriz] = acc
    return result


def get_iteration_files(folder: Path) -> list:
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    files = []
    for f in folder.iterdir():
        if pattern.match(f.name):
            files.append(f)
    return sorted(files, key=lambda f: int(pattern.match(f.name).group(1)))


def wilson_ci(p: float, n: int, confidence: float = 0.95) -> tuple:
    if n == 0 or np.isnan(p):
        return (float("nan"), float("nan"))
    z           = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre      = (p + z**2 / (2 * n)) / denominator
    margin      = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    lower = max(0.0, centre - margin)
    upper = min(1.0, centre + margin)
    return (lower, upper)

pipeline_rag = AdvancedRAG(
    path_vector_storage=PATH_VECTOR_STORAGE,
    embedding_type=EMBEDDING_TYPE,
    embedding_model_name=EMBEDDING_MODEL,
    cross_encoder_model_name=CROSS_ENCODER_MODEL_NAME,
    top_n=TOP_N
)

llm_evaluador = LLMEvaluator()

folder = Path(OUTPUT_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

last  = get_last_iteration(folder)
start = last + 1
end   = last + N_ITERATIONS

print(f"Iteraciones previas encontradas : {last}")
print(f"Corriendo iteraciones           : {start} → {end}")
print(f"Carpeta de salida               : {folder.resolve()}\n")

for i in range(start, end + 1):
    name_excel = folder / f"iteration_{i:03d}.xlsx"
    print(f"{'='*55}")
    print(f"  Iteración {i:03d} de {end:03d} → {name_excel.name}")
    print(f"{'='*55}")

    inicio = time.time()

    auto_evaluation_rag_v2(
        path_ground_truth=PATH_GROUND_TRUTH,
        name_excel=str(name_excel),
        pipeline_rag=pipeline_rag,
        llm_evaluador=llm_evaluador,
    )

    elapsed = time.time() - inicio
    print(f"  ✓ Completada en {elapsed:.1f}s ({elapsed/60:.2f} min)\n")

print("Todas las iteraciones completadas.")

Iteraciones previas encontradas : 0
Corriendo iteraciones           : 1 → 30
Carpeta de salida               : C:\Users\gol_m\OneDrive\Desktop\eswa\experiments\02_AdvancedRAG_TSF_STD_Evaluation

  Iteración 001 de 030 → iteration_001.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:42<00:00, 12.80s/it]


  ✓ Completada en 102.6s (1.71 min)

  Iteración 002 de 030 → iteration_002.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:15<00:00,  9.46s/it]


  ✓ Completada en 75.8s (1.26 min)

  Iteración 003 de 030 → iteration_003.xlsx


...

  ✓ Completada en 81.1s (1.35 min)

  Iteración 029 de 030 → iteration_029.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:23<00:00, 10.44s/it]


  ✓ Completada en 83.7s (1.39 min)

  Iteración 030 de 030 → iteration_030.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:17<00:00,  9.63s/it]

  ✓ Completada en 77.2s (1.29 min)

Todas las iteraciones completadas.


In [4]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

# Cargar todas las iteraciones
iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

# Construir DataFrame de detalle
all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

# Calcular estadísticas por matriz
stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

# Formatear detalle como porcentajes legibles
df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

# Mostrar en notebook
print("\nEstadísticas por matriz:")
display(df_stats)

# Exportar Excel
with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 30
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx
  iteration_021.xlsx
  iteration_022.xlsx
  iteration_023.xlsx
  iteration_024.xlsx
  iteration_025.xlsx
  iteration_026.xlsx
  iteration_027.xlsx
  iteration_028.xlsx
  iteration_029.xlsx
  iteration_030.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,30,50.00,0.00,33.15,66.85
1,Matriz 2,30,33.33,0.00,19.23,51.22
2,Matriz 4,30,73.33,44.98,55.55,85.82
3,Matriz 6,30,100.00,0.00,88.65,100.00
4,Matriz 7,30,41.67,18.95,25.98,59.25
5,Matriz 8,30,96.67,18.26,83.33,99.41
6,Total (matrices evaluadas),30,55.33,5.07,37.97,71.49



✓ Estadísticas exportadas → 02_AdvancedRAG_TSF_STD_Statistics.xlsx


### **2.2 Filtered Tailings Deposit (FTD)**

In [5]:
import os
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

N_ITERATIONS = 30

OUTPUT_FOLDER = "02_AdvancedRAG_TSF_FTD_Evaluation"

PATH_GROUND_TRUTH   = "../src/evaluation_rag/rag_ground_truth/02_FTD_Filtrado_GT.xlsx"
PATH_VECTOR_STORAGE = "../src/evaluation_retrieval/vector_storage/02_kozan_filtrado/02_kozan_nomic"
EMBEDDING_TYPE      = "nomic"
EMBEDDING_MODEL     = "nomic-embed-text-v1.5"

CROSS_ENCODER_MODEL_NAME="cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_N = 5

CONFIDENCE   = 0.95
OUTPUT_STATS = "02_AdvancedRAG_TSF_FTD_Statistics.xlsx"

def get_last_iteration(folder: Path) -> int:
    if not folder.exists():
        return 0
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    numbers = []
    for f in folder.iterdir():
        match = pattern.match(f.name)
        if match:
            numbers.append(int(match.group(1)))
    return max(numbers) if numbers else 0


def parse_accuracy(value) -> float:
    if isinstance(value, (int, float)):
        v = float(value)
        return v / 100 if v > 1 else v
    cleaned = str(value).replace("%", "").strip()
    try:
        v = float(cleaned)
        return v / 100 if v > 1 else v
    except ValueError:
        return float("nan")


def load_iteration(path: Path) -> dict:
    xl = pd.read_excel(path, sheet_name="resultado final")
    xl.columns = xl.columns.str.strip()
    result = {}
    for _, row in xl.iterrows():
        matriz = str(row["Matriz"]).strip()
        acc    = parse_accuracy(row["Accuracy"])
        result[matriz] = acc
    return result


def get_iteration_files(folder: Path) -> list:
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    files = []
    for f in folder.iterdir():
        if pattern.match(f.name):
            files.append(f)
    return sorted(files, key=lambda f: int(pattern.match(f.name).group(1)))


def wilson_ci(p: float, n: int, confidence: float = 0.95) -> tuple:
    if n == 0 or np.isnan(p):
        return (float("nan"), float("nan"))
    z           = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre      = (p + z**2 / (2 * n)) / denominator
    margin      = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    lower = max(0.0, centre - margin)
    upper = min(1.0, centre + margin)
    return (lower, upper)

pipeline_rag = AdvancedRAG(
    path_vector_storage=PATH_VECTOR_STORAGE,
    embedding_type=EMBEDDING_TYPE,
    embedding_model_name=EMBEDDING_MODEL,
    cross_encoder_model_name=CROSS_ENCODER_MODEL_NAME,
    top_n=TOP_N
)

llm_evaluador = LLMEvaluator()

folder = Path(OUTPUT_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

last  = get_last_iteration(folder)
start = last + 1
end   = last + N_ITERATIONS

print(f"Iteraciones previas encontradas : {last}")
print(f"Corriendo iteraciones           : {start} → {end}")
print(f"Carpeta de salida               : {folder.resolve()}\n")

for i in range(start, end + 1):
    name_excel = folder / f"iteration_{i:03d}.xlsx"
    print(f"{'='*55}")
    print(f"  Iteración {i:03d} de {end:03d} → {name_excel.name}")
    print(f"{'='*55}")

    inicio = time.time()

    auto_evaluation_rag_v2(
        path_ground_truth=PATH_GROUND_TRUTH,
        name_excel=str(name_excel),
        pipeline_rag=pipeline_rag,
        llm_evaluador=llm_evaluador,
    )

    elapsed = time.time() - inicio
    print(f"  ✓ Completada en {elapsed:.1f}s ({elapsed/60:.2f} min)\n")

print("Todas las iteraciones completadas.")

Iteraciones previas encontradas : 0
Corriendo iteraciones           : 1 → 30
Carpeta de salida               : C:\Users\gol_m\OneDrive\Desktop\eswa\experiments\02_AdvancedRAG_TSF_FTD_Evaluation

  Iteración 001 de 030 → iteration_001.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:58<00:00, 14.87s/it]


  ✓ Completada en 119.2s (1.99 min)

  Iteración 002 de 030 → iteration_002.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:51<00:00, 13.90s/it]


  ✓ Completada en 111.4s (1.86 min)

  Iteración 003 de 030 → iteration_003.xlsx


...

  ✓ Completada en 94.4s (1.57 min)

  Iteración 029 de 030 → iteration_029.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:31<00:00, 11.50s/it]


  ✓ Completada en 92.1s (1.54 min)

  Iteración 030 de 030 → iteration_030.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:36<00:00, 12.02s/it]

  ✓ Completada en 96.3s (1.60 min)

Todas las iteraciones completadas.


In [6]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

# Cargar todas las iteraciones
iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

# Construir DataFrame de detalle
all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

# Calcular estadísticas por matriz
stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

# Formatear detalle como porcentajes legibles
df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

# Mostrar en notebook
print("\nEstadísticas por matriz:")
display(df_stats)

# Exportar Excel
with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 30
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx
  iteration_021.xlsx
  iteration_022.xlsx
  iteration_023.xlsx
  iteration_024.xlsx
  iteration_025.xlsx
  iteration_026.xlsx
  iteration_027.xlsx
  iteration_028.xlsx
  iteration_029.xlsx
  iteration_030.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,30,100.00,0.00,88.65,100.00
1,Matriz 2,30,29.33,2.54,16.16,47.20
2,Matriz 4,30,0.00,0.00,0.00,11.35
3,Matriz 6,30,100.00,0.00,88.65,100.00
4,Matriz 7,30,50.00,0.00,33.15,66.85
5,Matriz 8,30,36.67,49.01,21.87,54.49
6,Total (matrices evaluadas),30,39.38,3.34,24.08,57.09



✓ Estadísticas exportadas → 02_AdvancedRAG_TSF_FTD_Statistics.xlsx


### **2.3 Thickened Tailings Deposit (TTD)**

In [7]:
import os
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

N_ITERATIONS = 30

OUTPUT_FOLDER = "02_AdvancedRAG_TSF_TTD_Evaluation"

PATH_GROUND_TRUTH   = "../src/evaluation_rag/rag_ground_truth/03_TTD_Espesado_GT.xlsx"
PATH_VECTOR_STORAGE = "../src/evaluation_retrieval/vector_storage/03_cenizas_espesado/03_cenizas_nomic"
EMBEDDING_TYPE      = "nomic"
EMBEDDING_MODEL     = "nomic-embed-text-v1.5"

CROSS_ENCODER_MODEL_NAME="cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_N = 5

CONFIDENCE   = 0.95
OUTPUT_STATS = "02_AdvancedRAG_TSF_TTD_Statistics.xlsx"

def get_last_iteration(folder: Path) -> int:
    if not folder.exists():
        return 0
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    numbers = []
    for f in folder.iterdir():
        match = pattern.match(f.name)
        if match:
            numbers.append(int(match.group(1)))
    return max(numbers) if numbers else 0


def parse_accuracy(value) -> float:
    if isinstance(value, (int, float)):
        v = float(value)
        return v / 100 if v > 1 else v
    cleaned = str(value).replace("%", "").strip()
    try:
        v = float(cleaned)
        return v / 100 if v > 1 else v
    except ValueError:
        return float("nan")


def load_iteration(path: Path) -> dict:
    xl = pd.read_excel(path, sheet_name="resultado final")
    xl.columns = xl.columns.str.strip()
    result = {}
    for _, row in xl.iterrows():
        matriz = str(row["Matriz"]).strip()
        acc    = parse_accuracy(row["Accuracy"])
        result[matriz] = acc
    return result


def get_iteration_files(folder: Path) -> list:
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    files = []
    for f in folder.iterdir():
        if pattern.match(f.name):
            files.append(f)
    return sorted(files, key=lambda f: int(pattern.match(f.name).group(1)))


def wilson_ci(p: float, n: int, confidence: float = 0.95) -> tuple:
    if n == 0 or np.isnan(p):
        return (float("nan"), float("nan"))
    z           = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre      = (p + z**2 / (2 * n)) / denominator
    margin      = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    lower = max(0.0, centre - margin)
    upper = min(1.0, centre + margin)
    return (lower, upper)

pipeline_rag = AdvancedRAG(
    path_vector_storage=PATH_VECTOR_STORAGE,
    embedding_type=EMBEDDING_TYPE,
    embedding_model_name=EMBEDDING_MODEL,
    cross_encoder_model_name=CROSS_ENCODER_MODEL_NAME,
    top_n=TOP_N
)

llm_evaluador = LLMEvaluator()

folder = Path(OUTPUT_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

last  = get_last_iteration(folder)
start = last + 1
end   = last + N_ITERATIONS

print(f"Iteraciones previas encontradas : {last}")
print(f"Corriendo iteraciones           : {start} → {end}")
print(f"Carpeta de salida               : {folder.resolve()}\n")

for i in range(start, end + 1):
    name_excel = folder / f"iteration_{i:03d}.xlsx"
    print(f"{'='*55}")
    print(f"  Iteración {i:03d} de {end:03d} → {name_excel.name}")
    print(f"{'='*55}")

    inicio = time.time()

    auto_evaluation_rag_v2(
        path_ground_truth=PATH_GROUND_TRUTH,
        name_excel=str(name_excel),
        pipeline_rag=pipeline_rag,
        llm_evaluador=llm_evaluador,
    )

    elapsed = time.time() - inicio
    print(f"  ✓ Completada en {elapsed:.1f}s ({elapsed/60:.2f} min)\n")

print("Todas las iteraciones completadas.")

Iteraciones previas encontradas : 0
Corriendo iteraciones           : 1 → 30
Carpeta de salida               : C:\Users\gol_m\OneDrive\Desktop\eswa\experiments\02_AdvancedRAG_TSF_TTD_Evaluation

  Iteración 001 de 030 → iteration_001.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:21<00:00, 10.20s/it]


  ✓ Completada en 81.8s (1.36 min)

  Iteración 002 de 030 → iteration_002.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:27<00:00, 11.00s/it]


  ✓ Completada en 88.2s (1.47 min)

  Iteración 003 de 030 → iteration_003.xlsx


...

  ✓ Completada en 68.5s (1.14 min)

  Iteración 029 de 030 → iteration_029.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:13<00:00,  9.15s/it]


  ✓ Completada en 73.4s (1.22 min)

  Iteración 030 de 030 → iteration_030.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:12<00:00,  9.07s/it]

  ✓ Completada en 72.8s (1.21 min)

Todas las iteraciones completadas.


In [8]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

# Cargar todas las iteraciones
iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

# Construir DataFrame de detalle
all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

# Calcular estadísticas por matriz
stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

# Formatear detalle como porcentajes legibles
df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

# Mostrar en notebook
print("\nEstadísticas por matriz:")
display(df_stats)

# Exportar Excel
with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 30
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx
  iteration_021.xlsx
  iteration_022.xlsx
  iteration_023.xlsx
  iteration_024.xlsx
  iteration_025.xlsx
  iteration_026.xlsx
  iteration_027.xlsx
  iteration_028.xlsx
  iteration_029.xlsx
  iteration_030.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,30,100.00,0.00,88.65,100.00
1,Matriz 2,30,38.33,7.77,23.22,56.09
2,Matriz 4,30,100.00,0.00,88.65,100.00
3,Matriz 6,30,100.00,0.00,88.65,100.00
4,Matriz 7,30,100.00,0.00,88.65,100.00
5,Matriz 8,30,50.00,50.85,33.15,66.85
6,Total (matrices evaluadas),30,65.00,5.54,47.14,79.46



✓ Estadísticas exportadas → 02_AdvancedRAG_TSF_TTD_Statistics.xlsx


### **2.4 Tailings Embankments (TE)**

In [9]:
import os
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

N_ITERATIONS = 30

OUTPUT_FOLDER = "02_AdvancedRAG_TSF_TE_Evaluation"

PATH_GROUND_TRUTH   = "../src/evaluation_rag/rag_ground_truth/04_TE_Embalse_GT.xlsx"
PATH_VECTOR_STORAGE = "../src/evaluation_retrieval/vector_storage/04_enami_embalse/04_enami_nomic"
EMBEDDING_TYPE      = "nomic"
EMBEDDING_MODEL     = "nomic-embed-text-v1.5"

CROSS_ENCODER_MODEL_NAME="cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_N = 5

CONFIDENCE   = 0.95
OUTPUT_STATS = "02_AdvancedRAG_TSF_TE_Statistics.xlsx"

def get_last_iteration(folder: Path) -> int:
    if not folder.exists():
        return 0
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    numbers = []
    for f in folder.iterdir():
        match = pattern.match(f.name)
        if match:
            numbers.append(int(match.group(1)))
    return max(numbers) if numbers else 0


def parse_accuracy(value) -> float:
    if isinstance(value, (int, float)):
        v = float(value)
        return v / 100 if v > 1 else v
    cleaned = str(value).replace("%", "").strip()
    try:
        v = float(cleaned)
        return v / 100 if v > 1 else v
    except ValueError:
        return float("nan")


def load_iteration(path: Path) -> dict:
    xl = pd.read_excel(path, sheet_name="resultado final")
    xl.columns = xl.columns.str.strip()
    result = {}
    for _, row in xl.iterrows():
        matriz = str(row["Matriz"]).strip()
        acc    = parse_accuracy(row["Accuracy"])
        result[matriz] = acc
    return result


def get_iteration_files(folder: Path) -> list:
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    files = []
    for f in folder.iterdir():
        if pattern.match(f.name):
            files.append(f)
    return sorted(files, key=lambda f: int(pattern.match(f.name).group(1)))


def wilson_ci(p: float, n: int, confidence: float = 0.95) -> tuple:
    if n == 0 or np.isnan(p):
        return (float("nan"), float("nan"))
    z           = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre      = (p + z**2 / (2 * n)) / denominator
    margin      = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    lower = max(0.0, centre - margin)
    upper = min(1.0, centre + margin)
    return (lower, upper)

pipeline_rag = AdvancedRAG(
    path_vector_storage=PATH_VECTOR_STORAGE,
    embedding_type=EMBEDDING_TYPE,
    embedding_model_name=EMBEDDING_MODEL,
    cross_encoder_model_name=CROSS_ENCODER_MODEL_NAME,
    top_n=TOP_N
)

llm_evaluador = LLMEvaluator()

folder = Path(OUTPUT_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

last  = get_last_iteration(folder)
start = last + 1
end   = last + N_ITERATIONS

print(f"Iteraciones previas encontradas : {last}")
print(f"Corriendo iteraciones           : {start} → {end}")
print(f"Carpeta de salida               : {folder.resolve()}\n")

for i in range(start, end + 1):
    name_excel = folder / f"iteration_{i:03d}.xlsx"
    print(f"{'='*55}")
    print(f"  Iteración {i:03d} de {end:03d} → {name_excel.name}")
    print(f"{'='*55}")

    inicio = time.time()

    auto_evaluation_rag_v2(
        path_ground_truth=PATH_GROUND_TRUTH,
        name_excel=str(name_excel),
        pipeline_rag=pipeline_rag,
        llm_evaluador=llm_evaluador,
    )

    elapsed = time.time() - inicio
    print(f"  ✓ Completada en {elapsed:.1f}s ({elapsed/60:.2f} min)\n")

print("Todas las iteraciones completadas.")

Iteraciones previas encontradas : 0
Corriendo iteraciones           : 1 → 30
Carpeta de salida               : C:\Users\gol_m\OneDrive\Desktop\eswa\experiments\02_AdvancedRAG_TSF_TE_Evaluation

  Iteración 001 de 030 → iteration_001.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:06<00:00,  8.32s/it]


  ✓ Completada en 66.8s (1.11 min)

  Iteración 002 de 030 → iteration_002.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:05<00:00,  8.16s/it]


  ✓ Completada en 65.5s (1.09 min)

  Iteración 003 de 030 → iteration_003.xlsx


...

  ✓ Completada en 66.9s (1.12 min)

  Iteración 029 de 030 → iteration_029.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:03<00:00,  7.97s/it]


  ✓ Completada en 63.9s (1.07 min)

  Iteración 030 de 030 → iteration_030.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:04<00:00,  8.03s/it]

  ✓ Completada en 64.4s (1.07 min)

Todas las iteraciones completadas.


In [10]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

# Cargar todas las iteraciones
iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

# Construir DataFrame de detalle
all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

# Calcular estadísticas por matriz
stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

# Formatear detalle como porcentajes legibles
df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

# Mostrar en notebook
print("\nEstadísticas por matriz:")
display(df_stats)

# Exportar Excel
with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 30
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx
  iteration_021.xlsx
  iteration_022.xlsx
  iteration_023.xlsx
  iteration_024.xlsx
  iteration_025.xlsx
  iteration_026.xlsx
  iteration_027.xlsx
  iteration_028.xlsx
  iteration_029.xlsx
  iteration_030.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,30,100.0,0.0,88.65,100.00
1,Matriz 2,30,75.0,0.0,57.30,87.02
2,Matriz 4,30,100.0,0.0,88.65,100.00
3,Matriz 6,30,0.0,0.0,0.00,11.35
4,Matriz 7,30,50.0,0.0,33.15,66.85
5,Matriz 8,30,100.0,0.0,88.65,100.00
6,Total (matrices evaluadas),30,70.0,0.0,52.12,83.34



✓ Estadísticas exportadas → 02_AdvancedRAG_TSF_TE_Statistics.xlsx


### **2.5 Paste Tailings Deposits (PTD)**

In [11]:
import os
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

N_ITERATIONS = 30

OUTPUT_FOLDER = "02_AdvancedRAG_TSF_PTD_Evaluation"

PATH_GROUND_TRUTH   = "../src/evaluation_rag/rag_ground_truth/05_PTD_EnPasta_GT.xlsx"
PATH_VECTOR_STORAGE = "../src/evaluation_retrieval/vector_storage/05_florida_pasta/05_florida_nomic"
EMBEDDING_TYPE      = "nomic"
EMBEDDING_MODEL     = "nomic-embed-text-v1.5"

CROSS_ENCODER_MODEL_NAME="cross-encoder/ms-marco-MiniLM-L-6-v2"
TOP_N = 5

CONFIDENCE   = 0.95
OUTPUT_STATS = "02_AdvancedRAG_TSF_PTD_Statistics.xlsx"

def get_last_iteration(folder: Path) -> int:
    if not folder.exists():
        return 0
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    numbers = []
    for f in folder.iterdir():
        match = pattern.match(f.name)
        if match:
            numbers.append(int(match.group(1)))
    return max(numbers) if numbers else 0


def parse_accuracy(value) -> float:
    if isinstance(value, (int, float)):
        v = float(value)
        return v / 100 if v > 1 else v
    cleaned = str(value).replace("%", "").strip()
    try:
        v = float(cleaned)
        return v / 100 if v > 1 else v
    except ValueError:
        return float("nan")


def load_iteration(path: Path) -> dict:
    xl = pd.read_excel(path, sheet_name="resultado final")
    xl.columns = xl.columns.str.strip()
    result = {}
    for _, row in xl.iterrows():
        matriz = str(row["Matriz"]).strip()
        acc    = parse_accuracy(row["Accuracy"])
        result[matriz] = acc
    return result


def get_iteration_files(folder: Path) -> list:
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    files = []
    for f in folder.iterdir():
        if pattern.match(f.name):
            files.append(f)
    return sorted(files, key=lambda f: int(pattern.match(f.name).group(1)))


def wilson_ci(p: float, n: int, confidence: float = 0.95) -> tuple:
    if n == 0 or np.isnan(p):
        return (float("nan"), float("nan"))
    z           = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre      = (p + z**2 / (2 * n)) / denominator
    margin      = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    lower = max(0.0, centre - margin)
    upper = min(1.0, centre + margin)
    return (lower, upper)

pipeline_rag = AdvancedRAG(
    path_vector_storage=PATH_VECTOR_STORAGE,
    embedding_type=EMBEDDING_TYPE,
    embedding_model_name=EMBEDDING_MODEL,
    cross_encoder_model_name=CROSS_ENCODER_MODEL_NAME,
    top_n=TOP_N
)

llm_evaluador = LLMEvaluator()

folder = Path(OUTPUT_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

last  = get_last_iteration(folder)
start = last + 1
end   = last + N_ITERATIONS

print(f"Iteraciones previas encontradas : {last}")
print(f"Corriendo iteraciones           : {start} → {end}")
print(f"Carpeta de salida               : {folder.resolve()}\n")

for i in range(start, end + 1):
    name_excel = folder / f"iteration_{i:03d}.xlsx"
    print(f"{'='*55}")
    print(f"  Iteración {i:03d} de {end:03d} → {name_excel.name}")
    print(f"{'='*55}")

    inicio = time.time()

    auto_evaluation_rag_v2(
        path_ground_truth=PATH_GROUND_TRUTH,
        name_excel=str(name_excel),
        pipeline_rag=pipeline_rag,
        llm_evaluador=llm_evaluador,
    )

    elapsed = time.time() - inicio
    print(f"  ✓ Completada en {elapsed:.1f}s ({elapsed/60:.2f} min)\n")

print("Todas las iteraciones completadas.")

Iteraciones previas encontradas : 0
Corriendo iteraciones           : 1 → 30
Carpeta de salida               : C:\Users\gol_m\OneDrive\Desktop\eswa\experiments\02_AdvancedRAG_TSF_PTD_Evaluation

  Iteración 001 de 030 → iteration_001.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:19<00:00,  9.88s/it]


  ✓ Completada en 79.3s (1.32 min)

  Iteración 002 de 030 → iteration_002.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:08<00:00,  8.54s/it]


  ✓ Completada en 68.5s (1.14 min)

  Iteración 003 de 030 → iteration_003.xlsx


...

  ✓ Completada en 81.7s (1.36 min)

  Iteración 029 de 030 → iteration_029.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [01:36<00:00, 12.05s/it]


  ✓ Completada en 96.6s (1.61 min)

  Iteración 030 de 030 → iteration_030.xlsx


Evaluación Matriz: 100%|██████████| 8/8 [04:00<00:00, 30.07s/it]

  ✓ Completada en 240.7s (4.01 min)

Todas las iteraciones completadas.


In [12]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

# Cargar todas las iteraciones
iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

# Construir DataFrame de detalle
all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

# Calcular estadísticas por matriz
stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

# Formatear detalle como porcentajes legibles
df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

# Mostrar en notebook
print("\nEstadísticas por matriz:")
display(df_stats)

# Exportar Excel
with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 30
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx
  iteration_021.xlsx
  iteration_022.xlsx
  iteration_023.xlsx
  iteration_024.xlsx
  iteration_025.xlsx
  iteration_026.xlsx
  iteration_027.xlsx
  iteration_028.xlsx
  iteration_029.xlsx
  iteration_030.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,30,100.00,0.00,88.65,100.00
1,Matriz 2,30,56.67,8.31,39.20,72.62
2,Matriz 4,30,100.00,0.00,88.65,100.00
3,Matriz 6,30,13.33,34.57,5.31,29.68
4,Matriz 7,30,50.00,0.00,33.15,66.85
5,Matriz 8,30,100.00,0.00,88.65,100.00
6,Total (matrices evaluadas),30,62.78,5.68,44.98,77.68



✓ Estadísticas exportadas → 02_AdvancedRAG_TSF_PTD_Statistics.xlsx


## **3. Hybrid RAG System Evaluation**

In [ ]:
from rag import HybridRAG

### **3.1 Sand Tailing Dams (STD)**

### **3.2 Filtered Tailings Deposit (FTD)**

### **3.3 Thickened Tailings Deposit (TTD)**

### **3.4 Tailings Embankments (TE)**

### **3.5 Paste Tailings Deposits (PTD)**

## **5. AsyncMultiQuery RAG System Evaluation**

In [1]:
from utils import EmbeddingConfig, AsyncMultiEmbeddingRAGV2
import os
import re
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from llms_modules import LLMEvaluator
from evaluation_rag import auto_evaluation_rag_v2

resource module not available on Windows


c:\Users\gol_m\anaconda3\envs\paper_geotecnia\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **1.1 Sand Tailing Dams (STD)**

In [2]:
N_ITERATIONS = 20

OUTPUT_FOLDER = "AsyncMultiQuery_TSF_STD_Evaluation"

PATH_GROUND_TRUTH = "../src/evaluation_rag/rag_ground_truth/01_STD_Tranque_GT.xlsx"

CONFIDENCE   = 0.95
OUTPUT_STATS = "AsyncMultiQuery_TSF_STD_Statistics.xlsx"

In [3]:
def get_last_iteration(folder: Path) -> int:
    if not folder.exists():
        return 0
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    numbers = []
    for f in folder.iterdir():
        match = pattern.match(f.name)
        if match:
            numbers.append(int(match.group(1)))
    return max(numbers) if numbers else 0


def parse_accuracy(value) -> float:
    if isinstance(value, (int, float)):
        v = float(value)
        return v / 100 if v > 1 else v
    cleaned = str(value).replace("%", "").strip()
    try:
        v = float(cleaned)
        return v / 100 if v > 1 else v
    except ValueError:
        return float("nan")


def load_iteration(path: Path) -> dict:
    xl = pd.read_excel(path, sheet_name="resultado final")
    xl.columns = xl.columns.str.strip()
    result = {}
    for _, row in xl.iterrows():
        matriz = str(row["Matriz"]).strip()
        acc    = parse_accuracy(row["Accuracy"])
        result[matriz] = acc
    return result


def get_iteration_files(folder: Path) -> list:
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    files = []
    for f in folder.iterdir():
        if pattern.match(f.name):
            files.append(f)
    return sorted(files, key=lambda f: int(pattern.match(f.name).group(1)))


def wilson_ci(p: float, n: int, confidence: float = 0.95) -> tuple:
    if n == 0 or np.isnan(p):
        return (float("nan"), float("nan"))
    z           = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre      = (p + z**2 / (2 * n)) / denominator
    margin      = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    lower = max(0.0, centre - margin)
    upper = min(1.0, centre + margin)
    return (lower, upper)

In [4]:
embedding_configs = [
    EmbeddingConfig(
        name="minilm",
        embedding_type="huggingface",
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        path_vector_storage="../src/evaluation_retrieval/vector_storage/01_cerro_negro_tranque/01_cerro_negro_all_mini_L6_v2"
    ),
    EmbeddingConfig(
        name="nomic",
        embedding_type="nomic",
        model_name="nomic-embed-text-v1.5",
        path_vector_storage="../src/evaluation_retrieval/vector_storage/01_cerro_negro_tranque/01_cerro_negro_nomic"
    )
]

pipeline_rag = AsyncMultiEmbeddingRAGV2(
    embedding_configs=embedding_configs,
    top_k=13,
    top_n=5
)

llm_evaluador = LLMEvaluator()

Configurando modelos de embeddings e índices...
Configurando embedding: minilm
✓ Embedding minilm configurado correctamente
Configurando embedding: nomic
✓ Embedding nomic configurado correctamente
Cargando Cross-Encoder...


In [5]:
folder = Path(OUTPUT_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

last  = get_last_iteration(folder)
start = last + 1
end   = last + N_ITERATIONS

print(f"Iteraciones previas encontradas : {last}")
print(f"Corriendo iteraciones           : {start} → {end}")
print(f"Carpeta de salida               : {folder.resolve()}\n")

for i in range(start, end + 1):
    name_excel = folder / f"iteration_{i:03d}.xlsx"
    print(f"{'='*55}")
    print(f"  Iteración {i:03d} de {end:03d} → {name_excel.name}")
    print(f"{'='*55}")

    inicio = time.time()

    auto_evaluation_rag_v2(
        path_ground_truth=PATH_GROUND_TRUTH,
        name_excel=str(name_excel),
        pipeline_rag=pipeline_rag,
        llm_evaluador=llm_evaluador,
    )

    elapsed = time.time() - inicio
    print(f"  ✓ Completada en {elapsed:.1f}s ({elapsed/60:.2f} min)\n")

print("Todas las iteraciones completadas.")

Iteraciones previas encontradas : 10
Corriendo iteraciones           : 11 → 30
Carpeta de salida               : C:\Users\gol_m\OneDrive\Desktop\eswa\experiments\AsyncMultiQuery_TSF_STD_Evaluation

  Iteración 011 de 030 → iteration_011.xlsx


Evaluación Matriz:   0%|          | 0/8 [00:00<?, ?it/s]

Matriz 1
Iniciando retrieval con múltiples embeddings...


✓ Retrieval completado para embedding: minilm


Procesando embeddings: 100%|██████████| 2/2 [00:02<00:00,  1.26s/it]

...

✓ Retrieval completado para embedding: minilm


Procesando embeddings: 100%|██████████| 2/2 [00:00<00:00,  4.10it/s]

✓ Retrieval completado para embedding: nomic
Agregando y filtrando nodos de todos los embeddings...
Procesando nodos de embedding: minilm
Procesando nodos de embedding: nomic
Total de nodos antes del filtrado: 130
Total de nodos después del filtrado: 46



Evaluación Matriz: 100%|██████████| 8/8 [04:08<00:00, 31.02s/it]

  ✓ Completada en 248.3s (4.14 min)

Todas las iteraciones completadas.


In [12]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

print("\nEstadísticas por matriz:")
display(df_stats)

with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 10
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,10,50.00,0.00,23.66,76.34
1,Matriz 2,10,66.67,0.00,36.78,87.30
2,Matriz 4,10,90.00,31.62,59.58,98.21
3,Matriz 6,10,100.00,0.00,72.25,100.00
4,Matriz 7,10,30.00,25.82,10.78,60.32
5,Matriz 8,10,100.00,0.00,72.25,100.00
6,Total (matrices evaluadas),10,65.00,7.07,35.37,86.31



✓ Estadísticas exportadas → AsyncMultiQuery_TSF_STD_Statistics.xlsx


In [6]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

print("\nEstadísticas por matriz:")
display(df_stats)

with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 30
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx
  iteration_021.xlsx
  iteration_022.xlsx
  iteration_023.xlsx
  iteration_024.xlsx
  iteration_025.xlsx
  iteration_026.xlsx
  iteration_027.xlsx
  iteration_028.xlsx
  iteration_029.xlsx
  iteration_030.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,30,50.00,0.00,33.15,66.85
1,Matriz 2,30,66.67,0.00,48.78,80.77
2,Matriz 4,30,76.67,43.02,59.07,88.21
3,Matriz 6,30,100.00,0.00,88.65,100.00
4,Matriz 7,30,23.33,25.37,11.79,40.93
5,Matriz 8,30,100.00,0.00,88.65,100.00
6,Total (matrices evaluadas),30,62.33,7.28,44.55,77.32



✓ Estadísticas exportadas → AsyncMultiQuery_TSF_STD_Statistics.xlsx


### **1.4 Tailing Embankments (TE)**

1hora 10 iteraciones.

In [6]:
N_ITERATIONS = 10

OUTPUT_FOLDER = "AsyncMultiQuery_TSF_TE_Evaluation"

PATH_GROUND_TRUTH = "../src/evaluation_rag/rag_ground_truth/04_TE_Embalse_GT.xlsx"

CONFIDENCE   = 0.95
OUTPUT_STATS = "AsyncMultiQuery_TSF_TE_Statistics.xlsx"

def get_last_iteration(folder: Path) -> int:
    if not folder.exists():
        return 0
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    numbers = []
    for f in folder.iterdir():
        match = pattern.match(f.name)
        if match:
            numbers.append(int(match.group(1)))
    return max(numbers) if numbers else 0


def parse_accuracy(value) -> float:
    if isinstance(value, (int, float)):
        v = float(value)
        return v / 100 if v > 1 else v
    cleaned = str(value).replace("%", "").strip()
    try:
        v = float(cleaned)
        return v / 100 if v > 1 else v
    except ValueError:
        return float("nan")


def load_iteration(path: Path) -> dict:
    xl = pd.read_excel(path, sheet_name="resultado final")
    xl.columns = xl.columns.str.strip()
    result = {}
    for _, row in xl.iterrows():
        matriz = str(row["Matriz"]).strip()
        acc    = parse_accuracy(row["Accuracy"])
        result[matriz] = acc
    return result


def get_iteration_files(folder: Path) -> list:
    pattern = re.compile(r"iteration_(\d+)\.xlsx$", re.IGNORECASE)
    files = []
    for f in folder.iterdir():
        if pattern.match(f.name):
            files.append(f)
    return sorted(files, key=lambda f: int(pattern.match(f.name).group(1)))


def wilson_ci(p: float, n: int, confidence: float = 0.95) -> tuple:
    if n == 0 or np.isnan(p):
        return (float("nan"), float("nan"))
    z           = stats.norm.ppf(1 - (1 - confidence) / 2)
    denominator = 1 + z**2 / n
    centre      = (p + z**2 / (2 * n)) / denominator
    margin      = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    lower = max(0.0, centre - margin)
    upper = min(1.0, centre + margin)
    return (lower, upper)

embedding_configs = [
    EmbeddingConfig(
        name="minilm",
        embedding_type="huggingface",
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        path_vector_storage="../src/evaluation_retrieval/vector_storage/04_enami_embalse/04_enami_all_mini_L6_v2"
    ),
    EmbeddingConfig(
        name="nomic",
        embedding_type="nomic",
        model_name="nomic-embed-text-v1.5",
        path_vector_storage="../src/evaluation_retrieval/vector_storage/04_enami_embalse/04_enami_nomic"
    )
]

pipeline_rag = AsyncMultiEmbeddingRAGV2(
    embedding_configs=embedding_configs,
    top_k=13,
    top_n=5
)

llm_evaluador = LLMEvaluator()

folder = Path(OUTPUT_FOLDER)
folder.mkdir(parents=True, exist_ok=True)

last  = get_last_iteration(folder)
start = last + 1
end   = last + N_ITERATIONS

print(f"Iteraciones previas encontradas : {last}")
print(f"Corriendo iteraciones           : {start} → {end}")
print(f"Carpeta de salida               : {folder.resolve()}\n")

for i in range(start, end + 1):
    name_excel = folder / f"iteration_{i:03d}.xlsx"
    print(f"{'='*55}")
    print(f"  Iteración {i:03d} de {end:03d} → {name_excel.name}")
    print(f"{'='*55}")

    inicio = time.time()

    auto_evaluation_rag_v2(
        path_ground_truth=PATH_GROUND_TRUTH,
        name_excel=str(name_excel),
        pipeline_rag=pipeline_rag,
        llm_evaluador=llm_evaluador,
    )

    elapsed = time.time() - inicio
    print(f"  ✓ Completada en {elapsed:.1f}s ({elapsed/60:.2f} min)\n")

print("Todas las iteraciones completadas.")

Configurando modelos de embeddings e índices...
Configurando embedding: minilm
✓ Embedding minilm configurado correctamente
Configurando embedding: nomic
✓ Embedding nomic configurado correctamente
Cargando Cross-Encoder...
Iteraciones previas encontradas : 20
Corriendo iteraciones           : 21 → 30
Carpeta de salida               : C:\Users\gol_m\OneDrive\Desktop\eswa\experiments\AsyncMultiQuery_TSF_TE_Evaluation

  Iteración 021 de 030 → iteration_021.xlsx


Evaluación Matriz:   0%|          | 0/8 [00:00<?, ?it/s]

Matriz 1
Iniciando retrieval con múltiples embeddings...


✓ Retrieval completado para embedding: minilm


Procesando embeddings: 100%|██████████| 2/2 [00:01<00:00,  1.92it/s]

...

✓ Retrieval completado para embedding: minilm


Procesando embeddings: 100%|██████████| 2/2 [00:00<00:00,  3.89it/s]

✓ Retrieval completado para embedding: nomic
Agregando y filtrando nodos de todos los embeddings...
Procesando nodos de embedding: minilm
Procesando nodos de embedding: nomic
Total de nodos antes del filtrado: 130
Total de nodos después del filtrado: 46



Evaluación Matriz: 100%|██████████| 8/8 [05:05<00:00, 38.18s/it]

  ✓ Completada en 305.6s (5.09 min)

Todas las iteraciones completadas.


In [3]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

print("\nEstadísticas por matriz:")
display(df_stats)

with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 10
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,10,100.0,0.00,72.25,100.00
1,Matriz 2,10,75.0,0.00,44.22,91.91
2,Matriz 4,10,10.0,31.62,1.79,40.42
3,Matriz 6,10,0.0,0.00,0.00,27.75
4,Matriz 7,10,25.0,26.35,8.09,55.78
5,Matriz 8,10,90.0,31.62,59.58,98.21
6,Total (matrices evaluadas),10,55.0,7.07,27.37,79.86



✓ Estadísticas exportadas → AsyncMultiQuery_TSF_TE_Statistics.xlsx


In [5]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

print("\nEstadísticas por matriz:")
display(df_stats)

with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 20
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,20,100.0,0.00,83.89,100.00
1,Matriz 2,20,75.0,0.00,53.13,88.81
2,Matriz 4,20,30.0,47.02,14.55,51.90
3,Matriz 6,20,0.0,0.00,0.00,16.11
4,Matriz 7,20,27.5,25.52,12.84,49.41
5,Matriz 8,20,95.0,22.36,76.39,99.11
6,Total (matrices evaluadas),20,58.0,8.34,36.86,76.56



✓ Estadísticas exportadas → AsyncMultiQuery_TSF_TE_Statistics.xlsx


In [7]:
folder = Path(OUTPUT_FOLDER)
files  = get_iteration_files(folder)

print(f"Archivos encontrados: {len(files)}")
for f in files:
    print(f"  {f.name}")

iterations_data = {}
for f in files:
    label = f.stem
    try:
        iterations_data[label] = load_iteration(f)
    except Exception as e:
        print(f"  ⚠ Error leyendo {f.name}: {e}")

all_matrices = []
for data in iterations_data.values():
    for m in data.keys():
        if m not in all_matrices:
            all_matrices.append(m)

detail_rows = []
for matriz in all_matrices:
    row = {"Matriz": matriz}
    for label, data in iterations_data.items():
        row[label] = data.get(matriz, float("nan"))
    detail_rows.append(row)

df_detail = pd.DataFrame(detail_rows)
iter_cols  = list(iterations_data.keys())

stats_rows = []
for _, row in df_detail.iterrows():
    values        = row[iter_cols].values.astype(float)
    values        = values[~np.isnan(values)]
    n             = len(values)
    mean          = np.mean(values) if n > 0 else float("nan")
    std           = np.std(values, ddof=1) if n > 1 else float("nan")
    ci_low, ci_up = wilson_ci(mean, n, CONFIDENCE)

    stats_rows.append({
        "Matriz":                           row["Matriz"],
        "N iteraciones":                    n,
        "Media (%)":                        round(mean   * 100, 2),
        "Std (%)":                          round(std    * 100, 2) if not np.isnan(std) else float("nan"),
        f"CI_lower {int(CONFIDENCE*100)}%": round(ci_low * 100, 2),
        f"CI_upper {int(CONFIDENCE*100)}%": round(ci_up  * 100, 2),
    })

df_stats = pd.DataFrame(stats_rows)

df_detail_pct = df_detail.copy()
for col in iter_cols:
    df_detail_pct[col] = df_detail_pct[col].apply(
        lambda x: f"{x*100:.2f}%" if not np.isnan(x) else "N/A"
    )

print("\nEstadísticas por matriz:")
display(df_stats)

with pd.ExcelWriter(OUTPUT_STATS, engine="openpyxl") as writer:
    df_detail_pct.to_excel(writer, sheet_name="Detalle",      index=False)
    df_stats.to_excel(     writer, sheet_name="Estadísticas", index=False)

print(f"\n✓ Estadísticas exportadas → {OUTPUT_STATS}")

Archivos encontrados: 30
  iteration_001.xlsx
  iteration_002.xlsx
  iteration_003.xlsx
  iteration_004.xlsx
  iteration_005.xlsx
  iteration_006.xlsx
  iteration_007.xlsx
  iteration_008.xlsx
  iteration_009.xlsx
  iteration_010.xlsx
  iteration_011.xlsx
  iteration_012.xlsx
  iteration_013.xlsx
  iteration_014.xlsx
  iteration_015.xlsx
  iteration_016.xlsx
  iteration_017.xlsx
  iteration_018.xlsx
  iteration_019.xlsx
  iteration_020.xlsx
  iteration_021.xlsx
  iteration_022.xlsx
  iteration_023.xlsx
  iteration_024.xlsx
  iteration_025.xlsx
  iteration_026.xlsx
  iteration_027.xlsx
  iteration_028.xlsx
  iteration_029.xlsx
  iteration_030.xlsx

Estadísticas por matriz:


,Matriz,N iteraciones,Media (%),Std (%),CI_lower 95%,CI_upper 95%
0,Matriz 1,30,100.00,0.00,88.65,100.00
1,Matriz 2,30,75.00,0.00,57.30,87.02
2,Matriz 4,30,26.67,44.98,14.18,44.45
3,Matriz 6,30,0.00,0.00,0.00,11.35
4,Matriz 7,30,25.00,25.43,12.98,42.70
5,Matriz 8,30,96.67,18.26,83.33,99.41
6,Total (matrices evaluadas),30,57.33,7.85,39.82,73.19



✓ Estadísticas exportadas → AsyncMultiQuery_TSF_TE_Statistics.xlsx
